In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

# 1. ĐỌC DỮ LIỆU
data = pd.read_csv(r'.\TrainingWiDS2021.csv')
target_col = 'diabetes_mellitus'

X = data.drop(columns=[target_col])
y = data[target_col]

# Chia tập Train/Test có phân tầng (stratify)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. TỰ ĐỘNG PHÂN LOẠI CÁC FEATURES
binary_cols = [col for col in X_train.columns if X_train[col].nunique() == 2]
categorical_cols = [col for col in X_train.select_dtypes(include=['object', 'category']).columns 
                    if col not in binary_cols]
numerical_cols = [col for col in X_train.select_dtypes(include=['int64', 'float64']).columns 
                  if col not in binary_cols]

# 3. XÂY DỰNG PIPELINE TIỀN XỬ LÝ (KHÔNG IMPUTE, KHÔNG SCALE)
# - Binary: Map về số, NaNs tự động thành -1
binary_transformer = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-1)

# - Categorical: One-Hot, bỏ qua giá trị lạ và NaN
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# - Numerical: Không biến đổi, để mô hình tự xử lý NaN
numerical_transformer = 'passthrough'

preprocessor = ColumnTransformer(transformers=[
    ('bin', binary_transformer, binary_cols),
    ('cat', categorical_transformer, categorical_cols),
    ('num', numerical_transformer, numerical_cols)
])

# 4. CHỌN MÔ HÌNH VÀ HUẤN LUYỆN
# HistGradientBoosting tự động xử lý missing values, hỗ trợ class_weight
model = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

pipeline.fit(X_train, y_train)

# 5. DỰ ĐOÁN VÀ ĐÁNH GIÁ
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

print("--- ĐÁNH GIÁ MÔ HÌNH HISTGRADIENTBOOSTING ---")
print("\n1. Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\n2. Classification Report:\n", classification_report(y_test, y_pred))
print(f"3. ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"4. PR-AUC (Average Precision): {average_precision_score(y_test, y_pred_proba):.4f}")

--- ĐÁNH GIÁ MÔ HÌNH HISTGRADIENTBOOSTING ---

1. Confusion Matrix:
 [[15997  4405]
 [ 1175  4455]]

2. Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.78      0.85     20402
           1       0.50      0.79      0.61      5630

    accuracy                           0.79     26032
   macro avg       0.72      0.79      0.73     26032
weighted avg       0.84      0.79      0.80     26032

3. ROC-AUC Score: 0.8639
4. PR-AUC (Average Precision): 0.6321


In [2]:
print(data.columns.to_list())

['Unnamed: 0', 'encounter_id', 'hospital_id', 'age', 'bmi', 'elective_surgery', 'ethnicity', 'gender', 'height', 'hospital_admit_source', 'icu_admit_source', 'icu_id', 'icu_stay_type', 'icu_type', 'pre_icu_los_days', 'readmission_status', 'weight', 'albumin_apache', 'apache_2_diagnosis', 'apache_3j_diagnosis', 'apache_post_operative', 'arf_apache', 'bilirubin_apache', 'bun_apache', 'creatinine_apache', 'fio2_apache', 'gcs_eyes_apache', 'gcs_motor_apache', 'gcs_unable_apache', 'gcs_verbal_apache', 'glucose_apache', 'heart_rate_apache', 'hematocrit_apache', 'intubated_apache', 'map_apache', 'paco2_apache', 'paco2_for_ph_apache', 'pao2_apache', 'ph_apache', 'resprate_apache', 'sodium_apache', 'temp_apache', 'urineoutput_apache', 'ventilated_apache', 'wbc_apache', 'd1_diasbp_invasive_max', 'd1_diasbp_invasive_min', 'd1_diasbp_max', 'd1_diasbp_min', 'd1_diasbp_noninvasive_max', 'd1_diasbp_noninvasive_min', 'd1_heartrate_max', 'd1_heartrate_min', 'd1_mbp_invasive_max', 'd1_mbp_invasive_min',

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression

# Thư viện cho 5 thuật toán và Tuning
import optuna
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

# --- 1. ĐỊNH NGHĨA CUSTOM PREPROCESSOR ---
class CustomWiDSPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=5):
        self.n_clusters = n_clusters
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)
        self.cluster_features = ['age', 'height', 'weight']
        
    def fit(self, X, y=None):
        X = X.copy()
        num_cols = X.select_dtypes(include=np.number).columns.tolist()
        if 'icu_id' in num_cols: num_cols.remove('icu_id')
        self.icu_aggs = X.groupby('icu_id')[num_cols].mean().add_suffix('_icu_mean')
        
        self.temp_impute_values = X[self.cluster_features].mean()
        X_cluster = X[self.cluster_features].fillna(self.temp_impute_values)
        clusters = self.kmeans.fit_predict(X_cluster)
        
        X['temp_cluster'] = clusters
        self.cluster_means = X.groupby('temp_cluster').mean(numeric_only=True)
        self.global_means = X.mean(numeric_only=True)
        return self

    def transform(self, X):
        X = X.copy()
        X = X.merge(self.icu_aggs, how='left', left_on='icu_id', right_index=True)
        cols_to_drop = ['icu_id', 'Unnamed: 0', 'encounter_id', 'hospital_id']
        X = X.drop(columns=[c for c in cols_to_drop if c in X.columns])
        
        X_cluster = X[self.cluster_features].fillna(self.temp_impute_values)
        clusters = self.kmeans.predict(X_cluster)
        for cluster_id in range(self.n_clusters):
            mask = (clusters == cluster_id)
            X.loc[mask] = X.loc[mask].fillna(self.cluster_means.loc[cluster_id])
            
        X = X.fillna(self.global_means)
        
        min_cols = [c for c in X.columns if c.endswith('_min')]
        for min_col in min_cols:
            max_col = min_col.replace('_min', '_max')
            if max_col in X.columns:
                base = min_col.replace('_min', '')
                X[f'{base}_mean_ext'] = (X[min_col] + X[max_col]) / 2
                X[f'{base}_range_ext'] = X[max_col] - X[min_col]
                X[f'{base}_ratio'] = X[max_col] / (X[min_col] + 1e-5)
        return X

# --- 2. ĐỌC VÀ CHIA DỮ LIỆU ---
data = pd.read_csv(r'.\TrainingWiDS2021.csv')
target_col = 'diabetes_mellitus'

X = data.drop(columns=[target_col])
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# --- 3. CHẠY CUSTOM PREPROCESSING ---
wids_prep = CustomWiDSPreprocessor(n_clusters=5)
X_tr_prep = wids_prep.fit_transform(X_tr)
X_val_prep = wids_prep.transform(X_val)
X_test_prep = wids_prep.transform(X_test)

binary_cols = [col for col in X_tr_prep.columns if X_tr_prep[col].nunique() == 2]
categorical_cols = [col for col in X_tr_prep.select_dtypes(include=['object', 'category']).columns if col not in binary_cols]
numerical_cols = [col for col in X_tr_prep.select_dtypes(include=['int64', 'float64']).columns if col not in binary_cols]

preprocessor = ColumnTransformer(transformers=[
    ('bin', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), binary_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols),
    ('num', 'passthrough', numerical_cols)
])

X_tr_final = preprocessor.fit_transform(X_tr_prep)
X_val_final = preprocessor.transform(X_val_prep)
X_test_final = preprocessor.transform(X_test_prep)

# --- 4. HÀM MỤC TIÊU OPTUNA ---
def objective(trial):
    algorithm = trial.suggest_categorical('algorithm', ['LightGBM', 'XGBoost', 'CatBoost', 'LogisticRegression', 'TabNet'])
    complexity = trial.suggest_categorical('complexity', ['shallow', 'medium', 'deep'])
    
    if complexity == 'shallow':
        max_depth = trial.suggest_int('max_depth', 3, 5)
        n_estimators = trial.suggest_int('n_estimators', 50, 150)
    elif complexity == 'medium':
        max_depth = trial.suggest_int('max_depth', 6, 8)
        n_estimators = trial.suggest_int('n_estimators', 200, 400)
    else:
        max_depth = trial.suggest_int('max_depth', 9, 12)
        n_estimators = trial.suggest_int('n_estimators', 500, 1000)

    try:
        if algorithm == 'LightGBM':
            model = lgb.LGBMClassifier(
                max_depth=max_depth, n_estimators=n_estimators, learning_rate=trial.suggest_float('lr', 1e-3, 0.1, log=True),
                class_weight='balanced', random_state=42, n_jobs=-1
            )
            model.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)])
            preds = model.predict_proba(X_val_final)[:, 1]

        elif algorithm == 'XGBoost':
            model = xgb.XGBClassifier(
                max_depth=max_depth, n_estimators=n_estimators, learning_rate=trial.suggest_float('lr', 1e-3, 0.1, log=True),
                scale_pos_weight=(len(y_tr) - sum(y_tr)) / sum(y_tr), random_state=42, n_jobs=-1, early_stopping_rounds=20
            )
            model.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)], verbose=False)
            preds = model.predict_proba(X_val_final)[:, 1]

        elif algorithm == 'CatBoost':
            model = CatBoostClassifier(
                depth=max_depth, iterations=n_estimators, learning_rate=trial.suggest_float('lr', 1e-3, 0.1, log=True),
                auto_class_weights='Balanced', random_seed=42, early_stopping_rounds=20, verbose=False
            )
            model.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)])
            preds = model.predict_proba(X_val_final)[:, 1]

        elif algorithm == 'TabNet':
            n_steps = 3 if complexity == 'shallow' else (5 if complexity == 'medium' else 7)
            model = TabNetClassifier(n_steps=n_steps, verbose=0, seed=42)
            model.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)], patience=15, max_epochs=100)
            preds = model.predict_proba(X_val_final)[:, 1]

        else: # Logistic Regression
            max_iter = 100 if complexity == 'shallow' else (300 if complexity == 'medium' else 800)
            model = LogisticRegression(
                C=trial.suggest_float('C', 1e-4, 10, log=True), class_weight='balanced',
                max_iter=max_iter, solver='saga', n_jobs=-1, random_state=42
            )
            model.fit(X_tr_final, y_tr)
            preds = model.predict_proba(X_val_final)[:, 1]
            
        return roc_auc_score(y_val, preds)
        
    except Exception as e:
        return 0.0

# --- 5. CHẠY TÌM KIẾM OPTUNA ---
study = optuna.create_study(
    direction='maximize', 
    pruner=optuna.pruners.HyperbandPruner(min_resource=10, max_resource='auto', reduction_factor=3)
)
print("Bắt đầu huấn luyện 120 models...")
study.optimize(objective, n_trials=120)

print(f"\nBest Trial: {study.best_trial.value}")
print(f"Best Params: {study.best_trial.params}")

# --- 6. HUẤN LUYỆN LẠI VÀ ĐÁNH GIÁ MÔ HÌNH TỐT NHẤT (ĐÃ SỬA) ---
print("\n--- ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST ---")
best_params = study.best_trial.params
best_algo = best_params['algorithm']
complexity = best_params['complexity']

# Tái tạo lại cấu hình từ Best Params
if best_algo == 'LightGBM':
    best_model = lgb.LGBMClassifier(
        max_depth=best_params['max_depth'], 
        n_estimators=best_params['n_estimators'], 
        learning_rate=best_params['lr'],
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    best_model.fit(X_tr_final, y_tr)
    
elif best_algo == 'XGBoost':
    best_model = xgb.XGBClassifier(
        max_depth=best_params['max_depth'], 
        n_estimators=best_params['n_estimators'], 
        learning_rate=best_params['lr'],
        scale_pos_weight=(len(y_tr) - sum(y_tr)) / sum(y_tr), 
        random_state=42, n_jobs=-1
    )
    best_model.fit(X_tr_final, y_tr)
    
elif best_algo == 'CatBoost':
    best_model = CatBoostClassifier(
        depth=best_params['max_depth'], 
        iterations=best_params['n_estimators'], 
        learning_rate=best_params['lr'],
        auto_class_weights='Balanced', random_seed=42, verbose=False
    )
    best_model.fit(X_tr_final, y_tr)
    
elif best_algo == 'TabNet':
    n_steps = 3 if complexity == 'shallow' else (5 if complexity == 'medium' else 7)
    best_model = TabNetClassifier(n_steps=n_steps, verbose=0, seed=42)
    # TabNet cần eval_set để có thể sử dụng early stopping (patience)
    best_model.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)], patience=15, max_epochs=100)
    
elif best_algo == 'LogisticRegression':
    max_iter = 100 if complexity == 'shallow' else (300 if complexity == 'medium' else 800)
    best_model = LogisticRegression(
        C=best_params['C'], class_weight='balanced',
        max_iter=max_iter, solver='saga', n_jobs=-1, random_state=42
    )
    best_model.fit(X_tr_final, y_tr)

# Đánh giá chung
y_pred_proba = best_model.predict_proba(X_test_final)[:, 1]
print(f"ROC-AUC Test Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
D:\ml_cache\temp\ipykernel_436\2485768659.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X[f'{base}_mean_ext'] = (X[min_col] + X[max_col]) / 2
D:\ml_cache\temp\ipykernel_436\2485768659.py:67: PerformanceWarn

--- ĐÁNH GIÁ MÔ HÌNH ---

1. Confusion Matrix:
 [[16078  4324]
 [ 1142  4488]]

2. Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.79      0.85     20402
           1       0.51      0.80      0.62      5630

    accuracy                           0.79     26032
   macro avg       0.72      0.79      0.74     26032
weighted avg       0.84      0.79      0.80     26032

3. ROC-AUC Score: 0.8683
4. PR-AUC (Average Precision): 0.6431


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression

# Thư viện cho 5 thuật toán và Tuning
import optuna
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

# --- 1. ĐỊNH NGHĨA CUSTOM PREPROCESSOR ---
class CustomWiDSPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=5):
        self.n_clusters = n_clusters
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)
        self.cluster_features = ['age', 'height', 'weight']
        
    def fit(self, X, y=None):
        X = X.copy()
        num_cols = X.select_dtypes(include=np.number).columns.tolist()
        if 'icu_id' in num_cols: num_cols.remove('icu_id')
        self.icu_aggs = X.groupby('icu_id')[num_cols].mean().add_suffix('_icu_mean')
        
        self.temp_impute_values = X[self.cluster_features].mean()
        X_cluster = X[self.cluster_features].fillna(self.temp_impute_values)
        clusters = self.kmeans.fit_predict(X_cluster)
        
        X['temp_cluster'] = clusters
        self.cluster_means = X.groupby('temp_cluster').mean(numeric_only=True)
        self.global_means = X.mean(numeric_only=True)
        return self

    def transform(self, X):
        X = X.copy()
        X = X.merge(self.icu_aggs, how='left', left_on='icu_id', right_index=True)
        cols_to_drop = ['icu_id', 'Unnamed: 0', 'encounter_id', 'hospital_id']
        X = X.drop(columns=[c for c in cols_to_drop if c in X.columns])
        
        X_cluster = X[self.cluster_features].fillna(self.temp_impute_values)
        clusters = self.kmeans.predict(X_cluster)
        for cluster_id in range(self.n_clusters):
            mask = (clusters == cluster_id)
            X.loc[mask] = X.loc[mask].fillna(self.cluster_means.loc[cluster_id])
            
        X = X.fillna(self.global_means)
        
        min_cols = [c for c in X.columns if c.endswith('_min')]
        for min_col in min_cols:
            max_col = min_col.replace('_min', '_max')
            if max_col in X.columns:
                base = min_col.replace('_min', '')
                X[f'{base}_mean_ext'] = (X[min_col] + X[max_col]) / 2
                X[f'{base}_range_ext'] = X[max_col] - X[min_col]
                X[f'{base}_ratio'] = X[max_col] / (X[min_col] + 1e-5)
        return X

# --- 2. ĐỌC VÀ CHIA DỮ LIỆU ---
data = pd.read_csv(r'.\TrainingWiDS2021.csv')
target_col = 'diabetes_mellitus'

X = data.drop(columns=[target_col])
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# --- 3. CHẠY CUSTOM PREPROCESSING ---
wids_prep = CustomWiDSPreprocessor(n_clusters=5)
X_tr_prep = wids_prep.fit_transform(X_tr)
X_val_prep = wids_prep.transform(X_val)
X_test_prep = wids_prep.transform(X_test)

binary_cols = [col for col in X_tr_prep.columns if X_tr_prep[col].nunique() == 2]
categorical_cols = [col for col in X_tr_prep.select_dtypes(include=['object', 'category']).columns if col not in binary_cols]
numerical_cols = [col for col in X_tr_prep.select_dtypes(include=['int64', 'float64']).columns if col not in binary_cols]

preprocessor = ColumnTransformer(transformers=[
    ('bin', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), binary_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols),
    ('num', 'passthrough', numerical_cols)
])

X_tr_final = preprocessor.fit_transform(X_tr_prep)
X_val_final = preprocessor.transform(X_val_prep)
X_test_final = preprocessor.transform(X_test_prep)

# --- 4. HÀM MỤC TIÊU OPTUNA ---
def objective(trial):
    algorithm = trial.suggest_categorical('algorithm', ['LightGBM', 'XGBoost', 'CatBoost', 'LogisticRegression', 'TabNet'])
    complexity = trial.suggest_categorical('complexity', ['shallow', 'medium', 'deep'])
    
    if complexity == 'shallow':
        max_depth = trial.suggest_int('max_depth', 3, 5)
        n_estimators = trial.suggest_int('n_estimators', 50, 150)
    elif complexity == 'medium':
        max_depth = trial.suggest_int('max_depth', 6, 8)
        n_estimators = trial.suggest_int('n_estimators', 200, 400)
    else:
        max_depth = trial.suggest_int('max_depth', 9, 12)
        n_estimators = trial.suggest_int('n_estimators', 500, 1000)

    try:
        if algorithm == 'LightGBM':
            model = lgb.LGBMClassifier(
                max_depth=max_depth, n_estimators=n_estimators, learning_rate=trial.suggest_float('lr', 1e-3, 0.1, log=True),
                class_weight='balanced', random_state=42, n_jobs=-1,
                device_type='gpu', verbose=-1  # Đã kích hoạt GPU và tắt log tránh crash
            )
            model.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)])
            preds = model.predict_proba(X_val_final)[:, 1]

        elif algorithm == 'XGBoost':
            model = xgb.XGBClassifier(
                max_depth=max_depth, n_estimators=n_estimators, learning_rate=trial.suggest_float('lr', 1e-3, 0.1, log=True),
                scale_pos_weight=(len(y_tr) - sum(y_tr)) / sum(y_tr), random_state=42, n_jobs=-1, early_stopping_rounds=20,
                tree_method='hist', device='cuda'  # Đã kích hoạt GPU
            )
            model.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)], verbose=False)
            preds = model.predict_proba(X_val_final)[:, 1]

        elif algorithm == 'CatBoost':
            model = CatBoostClassifier(
                depth=max_depth, iterations=n_estimators, learning_rate=trial.suggest_float('lr', 1e-3, 0.1, log=True),
                auto_class_weights='Balanced', random_seed=42, early_stopping_rounds=20, verbose=False,
                task_type='GPU'  # Đã kích hoạt GPU
            )
            model.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)])
            preds = model.predict_proba(X_val_final)[:, 1]

        elif algorithm == 'TabNet':
            n_steps = 3 if complexity == 'shallow' else (5 if complexity == 'medium' else 7)
            model = TabNetClassifier(n_steps=n_steps, verbose=0, seed=42)
            model.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)], patience=15, max_epochs=100)
            preds = model.predict_proba(X_val_final)[:, 1]

        else: # Logistic Regression
            max_iter = 100 if complexity == 'shallow' else (300 if complexity == 'medium' else 800)
            model = LogisticRegression(
                C=trial.suggest_float('C', 1e-4, 10, log=True), class_weight='balanced',
                max_iter=max_iter, solver='saga', n_jobs=-1, random_state=42
            )
            model.fit(X_tr_final, y_tr)
            preds = model.predict_proba(X_val_final)[:, 1]
            
        return roc_auc_score(y_val, preds)
        
    except Exception as e:
        return 0.0

# --- 5. CHẠY TÌM KIẾM OPTUNA ---
study = optuna.create_study(
    direction='maximize', 
    pruner=optuna.pruners.HyperbandPruner(min_resource=10, max_resource='auto', reduction_factor=3)
)
print("Bắt đầu huấn luyện 120 models...")
study.optimize(objective, n_trials=120)

print(f"\nBest Trial: {study.best_trial.value}")
print(f"Best Params: {study.best_trial.params}")

# --- 6. HUẤN LUYỆN LẠI VÀ ĐÁNH GIÁ MÔ HÌNH TỐT NHẤT ---
print("\n--- ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST ---")
best_params = study.best_trial.params
best_algo = best_params['algorithm']
complexity = best_params['complexity']

if best_algo == 'LightGBM':
    best_model = lgb.LGBMClassifier(
        max_depth=best_params['max_depth'], 
        n_estimators=best_params['n_estimators'], 
        learning_rate=best_params['lr'],
        class_weight='balanced', random_state=42, n_jobs=-1,
        device_type='gpu', verbose=-1  # Kích hoạt GPU
    )
    best_model.fit(X_tr_final, y_tr)
    
elif best_algo == 'XGBoost':
    best_model = xgb.XGBClassifier(
        max_depth=best_params['max_depth'], 
        n_estimators=best_params['n_estimators'], 
        learning_rate=best_params['lr'],
        scale_pos_weight=(len(y_tr) - sum(y_tr)) / sum(y_tr), 
        random_state=42, n_jobs=-1,
        tree_method='hist', device='cuda'  # Kích hoạt GPU
    )
    best_model.fit(X_tr_final, y_tr)
    
elif best_algo == 'CatBoost':
    best_model = CatBoostClassifier(
        depth=best_params['max_depth'], 
        iterations=best_params['n_estimators'], 
        learning_rate=best_params['lr'],
        auto_class_weights='Balanced', random_seed=42, verbose=False,
        task_type='GPU'  # Kích hoạt GPU
    )
    best_model.fit(X_tr_final, y_tr)
    
elif best_algo == 'TabNet':
    n_steps = 3 if complexity == 'shallow' else (5 if complexity == 'medium' else 7)
    best_model = TabNetClassifier(n_steps=n_steps, verbose=0, seed=42)
    best_model.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)], patience=15, max_epochs=100)
    
elif best_algo == 'LogisticRegression':
    max_iter = 100 if complexity == 'shallow' else (300 if complexity == 'medium' else 800)
    best_model = LogisticRegression(
        C=best_params['C'], class_weight='balanced',
        max_iter=max_iter, solver='saga', n_jobs=-1, random_state=42
    )
    best_model.fit(X_tr_final, y_tr)

# Đánh giá chung
y_pred_proba = best_model.predict_proba(X_test_final)[:, 1]
print(f"ROC-AUC Test Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
D:\ml_cache\temp\ipykernel_24292\901536868.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

Bắt đầu huấn luyện 120 models...


[I 2026-06-01 18:01:08,590] Trial 0 finished with value: 0.0 and parameters: {'algorithm': 'TabNet', 'complexity': 'deep', 'max_depth': 9, 'n_estimators': 752}. Best is trial 0 with value: 0.0.
[I 2026-06-01 18:01:08,880] Trial 1 finished with value: 0.0 and parameters: {'algorithm': 'LogisticRegression', 'complexity': 'medium', 'max_depth': 8, 'n_estimators': 386, 'C': 1.502226231570227}. Best is trial 0 with value: 0.0.
[I 2026-06-01 18:01:20,052] Trial 2 finished with value: 0.8146896677046419 and parameters: {'algorithm': 'CatBoost', 'complexity': 'shallow', 'max_depth': 3, 'n_estimators': 57, 'lr': 0.022621801394361003}. Best is trial 2 with value: 0.8146896677046419.
[I 2026-06-01 18:24:04,926] Trial 3 finished with value: 0.8544943894815415 and parameters: {'algorithm': 'CatBoost', 'complexity': 'deep', 'max_depth': 10, 'n_estimators': 620, 'lr': 0.004157235805721536}. Best is trial 3 with value: 0.8544943894815415.
[I 2026-06-01 18:24:05,337] Trial 4 finished with value: 0.0 an

[LightGBM] [Info] Number of positive: 18017, number of negative: 65283
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.510524 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 104534
[LightGBM] [Info] Number of data points in the train set: 83300, number of used features: 539
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-06-01 18:24:52,921] Trial 5 finished with value: 0.8748871034636695 and parameters: {'algorithm': 'LightGBM', 'complexity': 'medium', 'max_depth': 8, 'n_estimators': 281, 'lr': 0.04660061757076701}. Best is trial 5 with value: 0.8748871034636695.
[I 2026-06-01 18:41:08,178] Trial 6 finished with value: 0.8724903952921422 and parameters: {'algorithm': 'XGBoost', 'complexity': 'deep', 'max_depth': 11, 'n_estimators': 892, 'lr': 0.01726486436243541}. Best is trial 5 with value: 0.8748871034636695.


...

[LightGBM] [Info] Number of positive: 18017, number of negative: 65283
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.112243 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 104534
[LightGBM] [Info] Number of data points in the train set: 83300, number of used features: 539
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-06-01 20:27:39,894] Trial 55 finished with value: 0.8745567800879404 and parameters: {'algorithm': 'LightGBM', 'complexity': 'medium', 'max_depth': 7, 'n_estimators': 250, 'lr': 0.05231347377597411}. Best is trial 27 with value: 0.8759494382407653.
[I 2026-06-01 20:27:40,044] Trial 56 finished with value: 0.0 and parameters: {'algorithm': 'TabNet', 'complexity': 'shallow', 'max_depth': 3, 'n_estimators': 147}. Best is trial 27 with value: 0.8759494382407653.


[LightGBM] [Info] Number of positive: 18017, number of negative: 65283
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.095192 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 104534
[LightGBM] [Info] Number of data points in the train set: 83300, number of used features: 539
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


[W 2026-06-01 20:27:42,683] Trial 57 failed with parameters: {'algorithm': 'LightGBM', 'complexity': 'medium', 'max_depth': 7, 'n_estimators': 219, 'lr': 0.09881214163092066} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "D:\ml_cache\temp\ipykernel_24292\901536868.py", line 116, in objective
    model.fit(X_tr_final, y_tr, eval_set=[(X_val_final, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)])
  File "c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\lightgbm\sklearn.py", line 1560, in fit
    super().fit(
  File "c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\lightgbm\sklearn.py", line 1049, in fit
    self._Booster = train(
  File "c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\lightgbm\eng

KeyboardInterrupt: 